In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/sales_ca1_clean.parquet")
df = df.sort_values(["item_id", "date"]).reset_index(drop=True)
print(df.shape)

(4788267, 17)


In [2]:
# 1. Pour chaque produit : part des jours avec au moins 1 vente
p = df.groupby("item_id")["sales"].apply(lambda s: (s > 0).mean())

# 2. Seuil par produit : nombre de jours à 0 "trop long pour être du hasard"
seuil = np.ceil(np.log(0.01) / np.log(1 - p)).clip(lower=7)

# 3. Pour chaque ligne : longueur de la série de 0 dont elle fait partie
df["is_zero"] = df["sales"] == 0
df["bloc"] = df.groupby("item_id")["sales"].transform(lambda s: (s > 0).cumsum())
df["long_serie_0"] = df.groupby(["item_id", "bloc"])["is_zero"].transform("sum")

# 4. Rupture probable = jour à 0 dans une série plus longue que le seuil
df["seuil"] = df["item_id"].map(seuil)
df["rupture_probable"] = df["is_zero"] & (df["long_serie_0"] >= df["seuil"])

print("Lignes marquées rupture :", df["rupture_probable"].sum())
print("Part de toutes les lignes :", round(df["rupture_probable"].mean() * 100, 1), "%")
print("Part des 0 :", round(df.loc[df["is_zero"], "rupture_probable"].mean() * 100, 1), "%")
print("Produits concernés :", df.loc[df["rupture_probable"], "item_id"].nunique())

Lignes marquées rupture : 880032
Part de toutes les lignes : 18.4 %
Part des 0 : 33.3 %
Produits concernés : 2966
